In [134]:
# Ayhan Demir
# Project: HomeGuard System: a home security system using sensor readings.
random.seed(42)
import random

random.randint(30, 100)    # a random whole number from 30 to 100 → a temperature
random.random() < 0.3      # True about 30% of the time → "is there motion?"

from datetime import datetime

In [135]:
import datetime

In [136]:
# Create 4 sensors of the Peterson house with no current value at the end of the lists
def create_sensors():
    return [
        {"id": 1, "location": "Living Room", "type": "motion", "current_value": None},
        {"id": 2, "location": "Front Door", "type": "door", "current_value": None},
        {"id": 3, "location": "Kitchen", "type": "temperature", "current_value": None},
        {"id": 4, "location": "Bedroom", "type": "smoke", "current_value": None},
    ]

# Build  alert dictionary
def create_alert(time, severity, category, message):
    return {
        "time": time,
        "severity": severity,
        "category": category,
        "message": message,
    }

# Build the sensor list and set the house mode
sensors = create_sensors()
system_mode = "AWAY"

for sensor in sensors:
    print(sensor)


{'id': 1, 'location': 'Living Room', 'type': 'motion', 'current_value': None}
{'id': 2, 'location': 'Front Door', 'type': 'door', 'current_value': None}
{'id': 3, 'location': 'Kitchen', 'type': 'temperature', 'current_value': None}
{'id': 4, 'location': 'Bedroom', 'type': 'smoke', 'current_value': None}


In [137]:
# Limits as constants
FREEZE_TEMP = 35
HEAT_TEMP = 95
COMFORT_MIN = 65
COMFORT_MAX = 75

below section is done by me, but for structure and order, i ran via AI too

In [138]:
# Check one sensor against the rule table
# Returns an alert dictionary, or None if everything is fine
def check_sensor(sensor, mode):
    value = sensor["current_value"]
    location = sensor["location"]
    type = sensor["type"]
    now = datetime.datetime.now().strftime("%H:%M:%S")

    # Nothing read yet, so nothing to check
    if value is None:
        return None

    # Smoke: critical in every mode
    if type == "smoke":
        if value:
            return create_alert(now, "CRITICAL", "SAFETY", f"Smoke detected in {location}!")

    # Motion or door: only a problem when nobody is up and about
    elif type == "motion" or type == "door":
        if value:
            if type == "door":
                event = f"{location} opened"
            else:
                event = f"Motion detected in {location}"

            if mode == "AWAY":
                return create_alert(now, "HIGH", "SECURITY", f"{event} while in AWAY mode!")
            elif mode == "SLEEP":
                return create_alert(now, "MEDIUM", "SECURITY", f"{event} while in SLEEP mode!")

    # Temperature: safety first, comfort second
    elif type == "temperature":
        if value < FREEZE_TEMP:
            return create_alert(now, "MEDIUM", "SAFETY", f"{location} temperature is {value}°F, below {FREEZE_TEMP}°F!")
        elif value > HEAT_TEMP:
            return create_alert(now, "MEDIUM", "SAFETY", f"{location} temperature is {value}°F, above {HEAT_TEMP}°F!")
        elif mode == "HOME" and (value < COMFORT_MIN or value > COMFORT_MAX):
            return create_alert(now, "LOW", "COMFORT", f"{location} temperature is {value}°F, outside {COMFORT_MIN}-{COMFORT_MAX}°F")

    # No rule fired
    return None



In [139]:
# List where the log lines are saved
event_log = []


# Generate a reading: random gives the sensor a new value
def generate_reading(sensor):
    if sensor["type"] == "temperature":
        sensor["current_value"] = random.randint(30, 100)
    else:
        sensor["current_value"] = random.random() < 0.3


# Trigger an alert: print the [ALERT!] line
def trigger_alert(alert):
    print("[ALERT!]", alert["severity"] , alert["category"], alert["message"])


# Log the event: print the [LOG] line and save it in a list
def log_event(alert):
    line = "[LOG] [" + alert["time"] + "] Sending notification to homeowner..."
    print(line)
    event_log.append(line)


# Process the readings: read every sensor and check it with Step 3
def process_readings(sensors, mode):
    for sensor in sensors:
        generate_reading(sensor)
        alert = check_sensor(sensor, mode)
        if alert:
            trigger_alert(alert)
            log_event(alert)


In [140]:
#  Sensor class

class Sensor:
    # Set up a new sensor with the same 4 fields as the dictionary
    def __init__(self, id, location, type):
        self.id = id
        self.location = location
        self.type = type
        self.current_value = None

    # Make a new reading
    def read(self):
        if self.type == "temperature":
            self.current_value = random.randint(30, 100)
        else:
            self.current_value = random.random() < 0.3

    # Check the reading against the rule table
    # Returns an alert, or None if everything is fine
    def isAbnormal(self, mode):
        value = self.current_value
        location = self.location
        now = datetime.datetime.now().strftime("%H:%M:%S")

        # Smoke: critical in every mode
        if self.type == "smoke":
            if value:
                return create_alert(now, "CRITICAL", "SAFETY", f"Smoke detected in {location}!")

        # Motion or door: only a problem in AWAY mode
        elif self.type == "motion" or self.type == "door":
            if value and mode == "AWAY":
                if self.type == "door":
                    return create_alert(now, "HIGH", "SECURITY", f"{location} opened while in AWAY mode!")
                else:
                    return create_alert(now, "HIGH", "SECURITY", f"Motion detected in {location} while in AWAY mode!")

        # Temperature: safety in every mode, comfort only in HOME mode
        elif self.type == "temperature":
            if value < FREEZE_TEMP or value > HEAT_TEMP:
                return create_alert(now, "MEDIUM", "SAFETY", f"{location} temperature is {value}°F!")
            elif mode == "HOME" and (value < COMFORT_MIN or value > COMFORT_MAX):
                return create_alert(now, "LOW", "COMFORT", f"{location} temperature is {value}°F, outside {COMFORT_MIN}-{COMFORT_MAX}°F")

        # Everything is fine
        return None

    # Clear the value
    def reset(self):
        self.current_value = None


# Updated: create Sensor objects instead of dictionaries
def create_sensors():
    return [
        Sensor(1, "Living Room", "motion"),
        Sensor(2, "Front Door", "door"),
        Sensor(3, "Kitchen", "temperature"),
        Sensor(4, "Bedroom", "smoke"),
    ]


# Updated: use the Sensor methods
def process_readings(sensors, mode):
    for sensor in sensors:
        sensor.read()
        alert = sensor.isAbnormal(mode)
        if alert:
            trigger_alert(alert)
            log_event(alert)


sensors = create_sensors()

# Check: create one Sensor object, call read(), print its value
kitchen = Sensor(3, "Kitchen", "temperature")
kitchen.read()
print(kitchen.current_value)

65


at below cell, i created if and elif myself, reamaining is AI help

In [141]:
# Icon for each severity, as in the sample output
ICONS = {"CRITICAL": "🔥", "HIGH": "🚨", "MEDIUM": "⚠️", "LOW": "💡"}


# Updated: print the [ALERT!] line with its icon
def trigger_alert(alert):
    severity = alert["severity"]
    category = alert["category"]
    message = alert["message"]
    icon = ICONS[severity]
    print(f"[ALERT!] {icon} {severity}: {category}: {message}")


# Print one [READING] line in the sample format
def print_reading(sensor):
    value = sensor.current_value
    location = sensor.location

    if sensor.type == "motion":
        if value:
            print(f"[READING] {location} Motion: Motion detected")
        else:
            print(f"[READING] {location} Motion: No activity")

    elif sensor.type == "door":
        if value:
            print(f"[READING] {location}: OPENED")
        else:
            print(f"[READING] {location}: CLOSED")

    elif sensor.type == "temperature":
        if value < COMFORT_MIN or value > COMFORT_MAX:
            print(f"[READING] {location} Temperature: {value}°F (Out of range)")
        else:
            print(f"[READING] {location} Temperature: {value}°F (Normal)")

    elif sensor.type == "smoke":
        if value:
            print(f"[READING] {location} Smoke: DETECTED")
        else:
            print(f"[READING] {location} Smoke: CLEAR")


# Updated: read, print and check every sensor for one minute
def process_readings(sensors, mode, time_text):
    for sensor in sensors:
        sensor.read()
        print_reading(sensor)
        alert = sensor.isAbnormal(mode)
        if alert:
            alert["time"] = time_text
            trigger_alert(alert)
            log_event(alert)


# Main loop: 3 "minutes"
start_time = datetime.datetime.now().replace(second=0)

print("=== HomeGuard Security System ===")

for minute in range(3):
    current_time = start_time + datetime.timedelta(minutes=minute)
    time_text = current_time.strftime("%H:%M:%S")

    print(f"Time: {time_text}")
    if minute == 0:
        print(f"Mode: {system_mode}")
        print()

    process_readings(sensors, system_mode, time_text)
    print()

=== HomeGuard Security System ===
Time: 08:49:00
Mode: AWAY

[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [08:49:00] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [08:49:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 43°F (Out of range)
[READING] Bedroom Smoke: CLEAR

Time: 08:50:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [08:50:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 84°F (Out of range)
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom!
[LOG] [08:50:00] Sending notification to homeowner...

Time: 08:51:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in A

i did below steps first lines with AI, since i didnt know the command

In [142]:
# Step 7: Test the system

time_text = datetime.datetime.now().strftime("%H:%M:%S")

# Security test: run with system_mode = "AWAY"
system_mode = "AWAY"
print(f"=== Test in {system_mode} mode ===")
for minute in range(3):
    process_readings(sensors, system_mode, time_text)
    print()

# Comfort test: run with system_mode = "HOME"
system_mode = "HOME"
print(f"=== Test in {system_mode} mode ===")
for minute in range(3):
    process_readings(sensors, system_mode, time_text)
    print()

=== Test in AWAY mode ===
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 83°F (Out of range)
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom!
[LOG] [08:49:28] Sending notification to homeowner...

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 30°F (Out of range)
[ALERT!] ⚠️ MEDIUM: SAFETY: Kitchen temperature is 30°F!
[LOG] [08:49:28] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [08:49:28] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 65°F (Normal)
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom!
[LOG] [08:49:28] Sending notification to homeowner...

=== Test in HOME mode ===
[READING] Living Ro